# [Introduction to Data Science](http://datascience-intro.github.io/1MS041-2026/)    
## 1MS041, 2026 
&copy;2026 Raazesh Sainudiin, Benny Avelin. [Attribution 4.0 International     (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/)

# ProbSS 2 — Loading and checking public data

## What you will do

We will follow a saved copy of NOAA's Mauna Loa monthly carbon-dioxide data
from its documentation to a first plot. You will check who produced it, what
the columns mean, how missing values are marked, and when interpolated values
should be kept separate from observations.

Everything needed to run the notebook is stored locally. By the end, you will
have a source summary, a clear cleaning rule, one diagnostic plot, and a
caption that says what the plot can and cannot show.


In [ ]:
from hashlib import sha256

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pathlib import Path

def course_data(filename):
    candidates = (
        Path("data") / filename,
        Path("master/jp/data") / filename,
    )
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find {filename}. Run this notebook from master/jp "
        "or from the repository root."
    )


## 1. Start with the documentation

The bundled file header identifies NOAA ESRL, records a creation date of 6 December 2018, asks users to credit the data producers, and explains that earlier values can be revised. It says that -99.99 denotes a missing monthly average and -1 denotes a missing number of contributing daily means.

This repository copy is therefore a saved teaching copy, not the latest NOAA record. A current scientific analysis must verify the current source, citation, and reuse terms.


In [ ]:
co2_path = course_data("co2_mm_mlo.txt")
header = []
with co2_path.open(encoding="utf-8") as handle:
    for line in handle:
        if not line.startswith("#"):
            break
        header.append(line.rstrip())

print("Local SHA-256:", sha256(co2_path.read_bytes()).hexdigest())
for phrase in ("File Creation", "Missing months", "subject to change"):
    match = next((line for line in header if phrase.lower() in line.lower()), None)
    print(match)


## A source summary for this saved copy

- producer: NOAA ESRL, with some early observations credited in the header to Scripps Institution of Oceanography;
- unit: one calendar month at Mauna Loa;
- variables: year, month, decimal date, observed monthly average, interpolated average, seasonally adjusted trend, and number of contributing days;
- measurement unit: dry-air carbon-dioxide mole fraction in ppm;
- missing-value codes: -99.99 for the observed average and -1 for the contributing-day count;
- fitness: suitable for teaching loading, missingness, seasonality, and trend; not suitable as a current operational record;
- privacy: no person-level records;
- integrity: retain the file checksum and do not silently replace observed values by interpolated values.

### Check the source yourself

Complete this record without downloading data during notebook execution. If internet access is unavailable in the session, use the producer information and archived landing-page documentation supplied by the instructor, and mark which facts still require later verification.

1. Record the producer's official page, the dataset title, who produced it, the place and time period it covers, the measurement unit, and the date when you checked the page.
2. Summarise how the producer asks to be cited, what reuse is allowed, and whether the data may be revised. A public download does not automatically mean unrestricted reuse.
3. Write down the steps another analyst should follow to obtain the same data: the main page, file or API address, expected filename and format, retrieval date, and checksum. Never put passwords or private tokens in the notebook.
4. Name one other source you decided not to use and explain why—for example, its origin is unclear, it is out of date, its columns have changed, or its reuse terms are missing.
5. Give one scientific question these data can answer reasonably well and one they cannot.

The checksum and the loading code below let another person identify and read the same saved copy. They do not show that it is the producer's latest version.


## 2. Load the table and name the columns


In [ ]:
columns = [
    "year",
    "month",
    "decimal_date",
    "average",
    "interpolated",
    "trend",
    "days",
]
raw = pd.read_csv(
    co2_path,
    comment="#",
    sep=r"\s+",
    names=columns,
)

print(raw.head())
print("\nShape:", raw.shape)
print("\nDtypes:")
print(raw.dtypes)


In [ ]:
sentinel_average = int((raw["average"] == -99.99).sum())
sentinel_days = int((raw["days"] == -1).sum())

co2 = raw.copy()
co2.loc[co2["average"] == -99.99, "average"] = np.nan
co2.loc[co2["days"] == -1, "days"] = np.nan
co2["date"] = pd.to_datetime(
    {"year": co2["year"], "month": co2["month"], "day": 1}
)

assert co2["month"].between(1, 12).all()
assert not co2.duplicated(["year", "month"]).any()
assert co2["date"].is_monotonic_increasing
assert co2["interpolated"].notna().all()

quality_log = pd.DataFrame(
    {
        "check": [
            "rows loaded",
            "missing observed averages",
            "missing contributing-day counts",
            "duplicate year-month keys",
            "invalid month values",
        ],
        "result": [
            len(co2),
            sentinel_average,
            sentinel_days,
            int(co2.duplicated(["year", "month"]).sum()),
            int((~co2["month"].between(1, 12)).sum()),
        ],
        "action": [
            "retain",
            "represent as NaN; do not overwrite",
            "represent as NaN",
            "investigate if nonzero",
            "investigate if nonzero",
        ],
    }
)
quality_log


The observed average and the interpolated average are different variables. Interpolation can be useful, but replacing one by the other without a flag would erase information about missingness.


## 3. Take a first look


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(co2["date"], co2["average"], linewidth=1, label="observed monthly average")
ax.plot(co2["date"], co2["trend"], linewidth=2, label="seasonally adjusted trend")
missing = co2["average"].isna()
ax.scatter(
    co2.loc[missing, "date"],
    co2.loc[missing, "interpolated"],
    marker="x",
    color="black",
    label="interpolated where observed is missing",
)
ax.set(xlabel="date", ylabel="CO2 (ppm)", title="Mauna Loa monthly CO2 in the frozen course copy")
ax.legend()
ax.grid(alpha=0.2)
plt.show()


In [ ]:
co2["seasonal_component"] = co2["interpolated"] - co2["trend"]
monthly = co2.groupby("month")["seasonal_component"].agg(["mean", "std", "count"])

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.errorbar(
    monthly.index,
    monthly["mean"],
    yerr=monthly["std"],
    marker="o",
    capsize=3,
)
ax.axhline(0, color="black", linewidth=1)
ax.set(
    xlabel="calendar month",
    ylabel="interpolated minus trend (ppm)",
    title="Seasonal diagnostic; bars show one within-month standard deviation",
    xticks=range(1, 13),
)
ax.grid(alpha=0.2)
plt.show()


## Explain the plot in plain language

In this frozen NOAA teaching copy, the Mauna Loa monthly carbon-dioxide series rises over the recorded period and has a repeating seasonal component. Crosses identify months for which the file supplies an interpolated value instead of an observed monthly average. The plot describes this station and this historical copy; it does not by itself identify a cause, represent every location, or provide the latest NOAA values.


## Recap

Before you finish, make sure you can:

1. Explain how you found and checked the source. Include the official page, the date you checked it, citation and reuse information, clear download steps, a checksum, and one source you decided not to use.
2. Explain any corrections you made to the source summary or quality log after checking the official information.
3. Explain why -99.99 must not be included as an ordinary concentration value.
4. Give one analysis for which the interpolated column is useful and one for which preserving the missingness flag is essential.
5. Revise the caption for a named audience and retain the limitations.
